# Transformers — Self-Attention & Multi-Head Attention

> Based on Stanford CS230 — C5M4, Andrew Ng / DeepLearning.AI  
> Original paper: Vaswani et al. (2017) — *Attention Is All You Need*

The Transformer replaces recurrence entirely with **self-attention**, enabling full parallelisation and capturing long-range dependencies in $O(1)$ path length.

## Learning Objectives

1. Compute scaled dot-product self-attention from Q, K, V matrices
2. Explain the role of positional encoding
3. Describe multi-head attention and why multiple heads help
4. Understand the Transformer encoder block (attention + FFN + LayerNorm)

## Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- $Q \in \mathbb{R}^{n \times d_k}$ — queries
- $K \in \mathbb{R}^{n \times d_k}$ — keys  
- $V \in \mathbb{R}^{n \times d_v}$ — values
- Scaling by $\sqrt{d_k}$ prevents dot products from growing large in magnitude

## Multi-Head Attention

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

## Positional Encoding

Since attention is permutation-invariant, position is injected via sinusoidal signals:

$$PE_{(pos, 2i)}   = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

## Transformer Encoder Block

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 320 280" width="320" height="280" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="ta2" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Input -->
  <rect x="95" y="240" width="130" height="28" rx="4" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.5"/>
  <text x="160" y="258" text-anchor="middle" fill="#3a7bbf" font-size="10" font-weight="bold">Input + Pos. Enc.</text>
  <!-- Self-Attention -->
  <rect x="70" y="175" width="180" height="40" rx="4" fill="#f4b942" fill-opacity="0.18" stroke="#f4b942" stroke-width="1.5"/>
  <text x="160" y="196" text-anchor="middle" fill="#a07010" font-size="10" font-weight="bold">Multi-Head Self-Attention</text>
  <text x="160" y="208" text-anchor="middle" fill="#a07010" font-size="9">Attention(Q,K,V)</text>
  <!-- Add & Norm 1 -->
  <rect x="95" y="135" width="130" height="28" rx="4" fill="#7ecba1" fill-opacity="0.18" stroke="#7ecba1" stroke-width="1.4"/>
  <text x="160" y="153" text-anchor="middle" fill="#3d7a5e" font-size="10" font-weight="bold">Add &amp; LayerNorm</text>
  <!-- FFN -->
  <rect x="70" y="85" width="180" height="35" rx="4" fill="#c678dd" fill-opacity="0.15" stroke="#c678dd" stroke-width="1.5"/>
  <text x="160" y="103" text-anchor="middle" fill="#8a3ab8" font-size="10" font-weight="bold">Feed-Forward Network</text>
  <text x="160" y="114" text-anchor="middle" fill="#8a3ab8" font-size="9">FFN(x) = max(0,xW₁+b₁)W₂+b₂</text>
  <!-- Add & Norm 2 -->
  <rect x="95" y="45" width="130" height="28" rx="4" fill="#7ecba1" fill-opacity="0.18" stroke="#7ecba1" stroke-width="1.4"/>
  <text x="160" y="63" text-anchor="middle" fill="#3d7a5e" font-size="10" font-weight="bold">Add &amp; LayerNorm</text>
  <!-- Output -->
  <rect x="95" y="10" width="130" height="26" rx="4" fill="#e05c5c" fill-opacity="0.12" stroke="#e05c5c" stroke-width="1.4"/>
  <text x="160" y="26" text-anchor="middle" fill="#b03a3a" font-size="10" font-weight="bold">Output (to next block)</text>
  <!-- Arrows -->
  <line x1="160" y1="240" x2="160" y2="215" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta2)"/>
  <line x1="160" y1="175" x2="160" y2="163" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta2)"/>
  <line x1="160" y1="135" x2="160" y2="120" stroke="#aaa" stroke-width="1.2" marker-end="url(#ta2)"/>
  <line x1="160" y1="85"  x2="160" y2="73"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ta2)"/>
  <line x1="160" y1="45"  x2="160" y2="36"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ta2)"/>
  <!-- Skip connections -->
  <line x1="55" y1="240" x2="55" y2="149" stroke="#56b6c2" stroke-width="1.4" stroke-dasharray="4,3"/>
  <line x1="55" y1="149" x2="95" y2="149" stroke="#56b6c2" stroke-width="1.4" marker-end="url(#ta2)"/>
  <line x1="50" y1="175" x2="50" y2="59"  stroke="#56b6c2" stroke-width="1.4" stroke-dasharray="4,3"/>
  <line x1="50" y1="59"  x2="95" y2="59"  stroke="#56b6c2" stroke-width="1.4" marker-end="url(#ta2)"/>
  <text x="42" y="115" fill="#56b6c2" font-size="8" transform="rotate(-90,42,115)">residual</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

def softmax_rows(Z):
    E = np.exp(Z - Z.max(axis=-1, keepdims=True))
    return E / E.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax_rows(scores)
    return weights @ V, weights

def positional_encoding(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, None]
    i   = np.arange(d_model // 2)[None, :]
    angle = pos / 10000 ** (2 * i / d_model)
    PE[:, 0::2] = np.sin(angle)
    PE[:, 1::2] = np.cos(angle)
    return PE

# ---- Single-head self-attention ----
np.random.seed(42)
n, d_model, d_k = 8, 16, 8

X  = np.random.randn(n, d_model)  # sequence of n tokens
WQ = np.random.randn(d_model, d_k) * 0.1
WK = np.random.randn(d_model, d_k) * 0.1
WV = np.random.randn(d_model, d_k) * 0.1

Q = X @ WQ
K = X @ WK
V = X @ WV

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Input shape:      {X.shape}")
print(f"Q, K, V shapes:   {Q.shape}")
print(f"Output shape:     {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Row sums (should all be 1): {attn_weights.sum(axis=1).round(4)}")

# ---- Multi-head attention ----
def multi_head_attention(X, n_heads, d_k, d_v):
    d_model = X.shape[-1]
    np.random.seed(0)
    head_outputs = []
    all_weights  = []
    for h in range(n_heads):
        WQ = np.random.randn(d_model, d_k) * 0.1
        WK = np.random.randn(d_model, d_k) * 0.1
        WV = np.random.randn(d_model, d_v) * 0.1
        out, w = scaled_dot_product_attention(X @ WQ, X @ WK, X @ WV)
        head_outputs.append(out)
        all_weights.append(w)
    concat = np.concatenate(head_outputs, axis=-1)
    WO = np.random.randn(n_heads * d_v, d_model) * 0.1
    return concat @ WO, np.array(all_weights)

mha_out, all_weights = multi_head_attention(X, n_heads=4, d_k=4, d_v=4)

# ---- Plots ----
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Transformer — Self-Attention & Positional Encoding', fontsize=13, fontweight='bold')

tokens = [f'tok{i+1}' for i in range(n)]

# Self-attention weights heatmap
ax = axes[0, 0]
im = ax.imshow(attn_weights, cmap='Blues', vmin=0, vmax=attn_weights.max())
ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(tokens, fontsize=8)
ax.set_title('Self-Attention Weights (1 head)')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
plt.colorbar(im, ax=ax)

# Multi-head attention weights (4 heads)
for h in range(4):
    row, col = divmod(h, 2)
    ax = axes[row + (0 if h < 2 else 1), col + 1]
    im = ax.imshow(all_weights[h], cmap='Oranges', vmin=0, vmax=all_weights[h].max())
    ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(n)); ax.set_yticklabels(tokens, fontsize=7)
    ax.set_title(f'Head {h+1} attention', fontsize=10)
    plt.colorbar(im, ax=ax)

# Positional encoding heatmap
ax = axes[1, 0]
pe = positional_encoding(max_len=20, d_model=32)
im = ax.imshow(pe.T, cmap='RdYlBu', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('Position'); ax.set_ylabel('Encoding dimension')
ax.set_title('Positional Encoding (20 × 32)')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

# ---- Positional encoding curves ----
fig, ax = plt.subplots(figsize=(11, 4))
pe50 = positional_encoding(50, 64)
for dim, color in [(0, P[0]), (1, P[1]), (4, P[2]), (8, P[3])]:
    ax.plot(pe50[:, dim], color=color, lw=1.8, label=f'dim {dim}')
ax.set_xlabel('Position'); ax.set_ylabel('PE value')
ax.set_title('Positional Encoding — Sinusoidal Curves for Selected Dimensions')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.show()
